# Tahap 3 — Data Preprocessing & Preparation
Semua analisis EDA memakai data **train 2024** saja. Data 2025 (OOT) hanya di-clean dan disimpan.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config
from src.data_prep import (
    clean, make_target, missingness_by_target, save_processed, split_frames, stratified_split,
)
from src.features import add_engineered
from src.reporting import save_figure, save_table, set_plot_style

set_plot_style()
LBL = config.FEATURE_LABELS
raw = {y: pd.read_parquet(config.DATA_INTERIM / f"hmda_filtered_{y}.parquet") for y in (config.YEAR_TRAIN, config.YEAR_OOT)}

## 1. Target

In [2]:
y_year = {yr: make_target(df) for yr, df in raw.items()}
for yr, y in y_year.items():
    print(yr, "n =", len(y), "| ditolak =", int(y.sum()), f"({y.mean():.2%})")

2024 n = 55002 | ditolak = 6464 (11.75%)
2025 n = 53241 | ditolak = 5150 (9.67%)


## 2. Audit leakage (data 2024)
Kolom pasca-keputusan dibuang lebih dulu; sisa kolom diuji pola missing per kelas target.

In [3]:
c24 = clean(raw[config.YEAR_TRAIN])
y24 = y_year[config.YEAR_TRAIN]
audit_cols = [c for c in c24.columns if c not in config.LEAKAGE_COLS + ["action_taken"]]
leak = missingness_by_target(c24, y24, audit_cols)
leak = leak.reindex(leak["selisih_pp"].abs().sort_values(ascending=False).index)
save_table(leak, "t3_leakage_test", decimals=2)
leak.head(12)

,kolom,pct_na_y0,pct_na_y1,selisih_pp,n_kosong,pct_ditolak_bila_kosong,lift,flag_leakage
18,loan_to_value_ratio,2.519675,52.042079,49.522404,4587,73.337693,6.240284,True
55,co-applicant_age_above_62,45.578722,60.102104,14.523382,26008,14.937711,1.271046,False
25,applicant_ethnicity-2,93.273312,79.857673,-13.415638,50435,10.234956,0.870890,False
5,census_tract,0.076229,9.931931,9.855702,679,94.550810,8.045303,True
20,property_value,0.972434,8.183787,7.211353,1001,52.847153,4.496750,True
30,co-applicant_ethnicity-2,96.670650,92.527847,-4.142803,52903,11.305597,0.961990,False
62,dti_mid,0.739627,3.063119,2.323492,557,35.547576,3.024734,True
23,debt_to_income_ratio,0.739627,3.063119,2.323492,557,35.547576,3.024734,True
22,income,1.087807,3.341584,2.253777,744,29.032258,2.470347,True
42,co-applicant_race-2,97.420578,98.313738,0.893160,53641,11.847281,1.008082,False


Uji tambahan: kode kategori yang hampir menentukan target (artefak pelaporan/proses).

In [4]:
code_rows = []
for col in ["preapproval", "aus-1", "initially_payable_to_institution", "submission_of_application"]:
    g = pd.DataFrame({"kode": c24[col].fillna("kosong"), "y": y24}).groupby("kode")["y"].agg(["size", "mean"])
    for kode, r in g.iterrows():
        code_rows.append({"kolom": col, "kode": kode, "n": int(r["size"]), "pct_ditolak": r["mean"] * 100})
code_test = pd.DataFrame(code_rows)
save_table(code_test, "t3_leakage_code_test", decimals=2)
code_test

,kolom,kode,n,pct_ditolak
0,preapproval,1,2298,0.000000
1,preapproval,2,52704,12.264724
2,aus-1,1,28315,5.015010
3,aus-1,1111,1,0.000000
4,aus-1,2,17100,4.584795
5,aus-1,3,24,20.833333
6,aus-1,5,126,14.285714
7,aus-1,6,9056,46.322880
8,aus-1,7,380,11.052632
9,initially_payable_to_institution,1,50229,11.268391


In [5]:
ltv_calc = c24["loan_amount"] / c24["property_value"] * 100
ltv_check = pd.DataFrame({
    "fitur": ["loan_to_value_ratio (laporan)", "ltv_calc (hitungan)"],
    "pct_na_y0": [c24.loc[y24 == 0, "loan_to_value_ratio"].isna().mean() * 100, ltv_calc[y24 == 0].isna().mean() * 100],
    "pct_na_y1": [c24.loc[y24 == 1, "loan_to_value_ratio"].isna().mean() * 100, ltv_calc[y24 == 1].isna().mean() * 100],
})
both = ltv_calc.notna() & c24["loan_to_value_ratio"].notna()
ltv_check["korelasi_dengan_ltv_laporan"] = [1.0, np.corrcoef(ltv_calc[both], pd.to_numeric(c24.loc[both, "loan_to_value_ratio"]))[0, 1]]
save_table(ltv_check, "t3_ltv_replacement", decimals=3)
ltv_check

,fitur,pct_na_y0,pct_na_y1,korelasi_dengan_ltv_laporan
0,loan_to_value_ratio (laporan),2.519675,52.042079,1.000000
1,ltv_calc (hitungan),0.972434,8.183787,0.948368


## 3. Cleaning + feature engineering (kedua tahun) dan pemeriksaan kualitas data

In [6]:
quality, frames = [], {}
for yr, df in raw.items():
    dup = int(df.duplicated().sum())
    inc_raw = pd.to_numeric(df["income"], errors="coerce")
    cl = add_engineered(clean(df))
    frames[yr] = split_frames(cl)
    quality.append({
        "tahun": yr, "n": len(df), "duplikat_persis": dup,
        "income_nonpositif": int((inc_raw <= 0).sum()),
        "ltv_calc_di_atas_150": int((cl["ltv_calc"] > 150).sum()),
        "loan_lebih_besar_2x_properti": int((cl["loan_amount"] > 2 * cl["property_value"]).sum()),
        "dti_exempt_atau_na": int(cl["dti_mid"].isna().sum()),
    })
quality = pd.DataFrame(quality)
save_table(quality, "t3_data_quality")
quality

,tahun,n,duplikat_persis,income_nonpositif,ltv_calc_di_atas_150,loan_lebih_besar_2x_properti,dti_exempt_atau_na
0,2024,55002,11,331,52,18,557
1,2025,53241,13,178,46,18,630


## 4. Split: 2024 → train/test stratified 80/20; 2025 → OOT utuh

In [7]:
X24, s24, m24 = frames[config.YEAR_TRAIN]
parts = stratified_split(X24, y24, s24, m24)
for name, (X, y, s, m) in parts.items():
    save_processed(name, X, y, s, m)
X25, s25, m25 = frames[config.YEAR_OOT]
save_processed("oot", X25, y_year[config.YEAR_OOT], s25, m25)
X_train, y_train = parts["train"][0], parts["train"][1]

balance = pd.DataFrame([
    {"split": n, "n": len(y), "n_ditolak": int(y.sum()), "n_disetujui": int((y == 0).sum()), "pct_ditolak": y.mean() * 100}
    for n, y in (("train 2024", parts["train"][1]), ("test 2024", parts["test"][1]), ("OOT 2025", y_year[config.YEAR_OOT]))
])
balance["rasio_negatif_per_positif"] = balance["n_disetujui"] / balance["n_ditolak"]
save_table(balance, "t3_class_balance", decimals=2)
balance

,split,n,n_ditolak,n_disetujui,pct_ditolak,rasio_negatif_per_positif
0,train 2024,44001,5171,38830,11.752006,7.509186
1,test 2024,11001,1293,9708,11.753477,7.508121
2,OOT 2025,53241,5150,48091,9.672996,9.338058


## 5. Pemeriksaan set fitur final (evidence)

In [8]:
feats = list(X_train.columns)
assert set(feats).isdisjoint(config.LEAKAGE_COLS + config.PROTECTED_COLS + config.PROXY_COLS + config.LEAKAGE_AUDIT_DROPPED)
save_table(pd.DataFrame({
    "fitur": feats,
    "label": [LBL[f] for f in feats],
    "jenis": ["numerik" if f in config.NUMERIC_FEATURES else "kategorikal" for f in feats],
    "log_transform": [f in config.LOG_FEATURES for f in feats],
}), "t3_final_features")
print(len(feats), "fitur:", feats)

13 fitur: ['loan_amount', 'income', 'property_value', 'ltv_calc', 'dti_mid', 'loan_term', 'tract_to_msa_income_percentage', 'ffiec_msa_md_median_family_income', 'loan_to_income', 'property_to_income', 'submission_of_application', 'conforming_loan_limit', 'has_coapplicant']


## 6. Ringkasan missing value (train)

In [9]:
miss = (X_train.isna().mean() * 100).rename("pct_missing_train").to_frame()
miss["pct_missing_test"] = parts["test"][0].isna().mean() * 100
miss = miss.reset_index(names="fitur").sort_values("pct_missing_train", ascending=False)
save_table(miss, "t3_missing_summary", decimals=2)

fig, ax = plt.subplots(figsize=(7, 4))
m = miss[miss["pct_missing_train"] > 0]
ax.barh([LBL[f] for f in m["fitur"]], m["pct_missing_train"], color="#4C72B0")
ax.invert_yaxis()
ax.set_xlabel("Data kosong pada data latih (%)")
ax.set_ylabel("")
save_figure(fig, "f3_missing_summary")
miss

,fitur,pct_missing_train,pct_missing_test
9,property_to_income,3.061294,2.990637
3,ltv_calc,1.829504,1.781656
2,property_value,1.829504,1.781656
8,loan_to_income,1.352242,1.354422
1,income,1.352242,1.354422
4,dti_mid,1.015886,0.999909
5,loan_term,0.074998,0.081811
0,loan_amount,0.000000,0.000000
6,tract_to_msa_income_percentage,0.000000,0.000000
7,ffiec_msa_md_median_family_income,0.000000,0.000000


## 7. EDA (data latih)

In [10]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar([str(y) for y in y_year], [y.mean() * 100 for y in y_year.values()], color=["#4C72B0", "#DD8452"])
for i, y in enumerate(y_year.values()):
    ax.text(i, y.mean() * 100 + 0.2, f"{y.mean() * 100:.2f}".replace(".", ","), ha="center")
ax.set_xlabel("Tahun data")
ax.set_ylabel("Aplikasi ditolak (%)")
save_figure(fig, "f3_target_per_tahun")

PosixPath('/home/sevima-375/Binus/Exam/Business Intelligence & Analytics/hmda/reports/figures/f3_target_per_tahun.png')

In [11]:
from src.reporting import fmt_id


def bin_label(b) -> str:
    if isinstance(b, pd.Interval):
        if np.isinf(b.right):
            return f"> {fmt_id(b.left, 0)}"
        return f"{fmt_id(b.left, 0)}–{fmt_id(b.right, 0)}"
    return str(b)


def denial_bar(series: pd.Series, y: pd.Series, xlabel: str, name: str) -> pd.DataFrame:
    g = pd.DataFrame({"bin": series, "y": y}).dropna().groupby("bin", observed=True, sort=True)["y"].agg(["size", "mean"])
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.bar([bin_label(b) for b in g.index], g["mean"] * 100, color="#4C72B0")
    ax.axhline(y.mean() * 100, color="#C44E52", linestyle="--", linewidth=1, label="Rata-rata data latih")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Tingkat penolakan (%)")
    ax.legend()
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    save_figure(fig, name)
    return g


dti_order = [str(int(v)) for v in sorted(X_train["dti_mid"].dropna().unique())] + ["Kosong"]
dti_bin = pd.Categorical(X_train["dti_mid"].map(lambda v: "Kosong" if pd.isna(v) else str(int(v))), categories=dti_order, ordered=True)
denial_bar(pd.Series(dti_bin, index=X_train.index), y_train, LBL["dti_mid"] + " — nilai tengah rentang", "f3_denial_rate_dti")
denial_bar(pd.cut(X_train["ltv_calc"], [0, 60, 70, 80, 85, 90, 95, 100, np.inf]), y_train, LBL["ltv_calc"], "f3_denial_rate_ltv")
denial_bar(pd.qcut(X_train["income"], 5, precision=0), y_train, LBL["income"] + " — kuintil", "f3_denial_rate_income")
denial_bar(pd.qcut(X_train["loan_amount"], 5, precision=0), y_train, LBL["loan_amount"] + " — kuintil", "f3_denial_rate_loan_amount")

,size,mean
bin,,
"(4999.0, 265000.0]",9237,0.342860
"(265000.0, 385000.0]",8575,0.068688
"(385000.0, 495000.0]",9078,0.050892
"(495000.0, 645000.0]",8567,0.050660
"(645000.0, 16005000.0]",8544,0.060744


In [12]:
lo, hi = X_train[config.LOG_FEATURES].quantile(0.01), X_train[config.LOG_FEATURES].quantile(0.99)
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for j, col in enumerate(config.LOG_FEATURES):
    v = X_train[col].dropna()
    axes[0, j].hist(v, bins=50, color="#4C72B0")
    axes[0, j].xaxis.set_major_locator(plt.MaxNLocator(3))
    axes[0, j].set_xlabel(LBL[col] + " (asli)")
    axes[1, j].hist(np.log1p(v.clip(lo[col], hi[col])), bins=50, color="#55A868")
    axes[1, j].set_xlabel(LBL[col] + " (capping 1–99 + log)")
    axes[0, j].set_ylabel("Frekuensi")
    axes[1, j].set_ylabel("Frekuensi")
fig.tight_layout()
save_figure(fig, "f3_dist_before_after_log")

PosixPath('/home/sevima-375/Binus/Exam/Business Intelligence & Analytics/hmda/reports/figures/f3_dist_before_after_log.png')

In [13]:
num = [c for c in config.NUMERIC_FEATURES]
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, col in zip(axes.ravel(), num):
    ax.boxplot(X_train[col].dropna(), vert=True, widths=0.5, flierprops={"markersize": 2})
    ax.set_xticks([])
    ax.set_ylabel(LBL[col], fontsize=8)
fig.tight_layout()
save_figure(fig, "f3_boxplot_numerik")

outlier = pd.DataFrame({
    "fitur": num,
    "p1": [X_train[c].quantile(0.01) for c in num],
    "median": [X_train[c].median() for c in num],
    "p99": [X_train[c].quantile(0.99) for c in num],
    "maks": [X_train[c].max() for c in num],
    "pct_di_atas_p99": [(X_train[c] > X_train[c].quantile(0.99)).mean() * 100 for c in num],
})
save_table(outlier, "t3_outlier_percentiles", decimals=2)
outlier

/tmp/ipykernel_181896/4084855955.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(X_train[col].dropna(), vert=True, widths=0.5, flierprops={"markersize": 2})
/tmp/ipykernel_181896/4084855955.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(X_train[col].dropna(), vert=True, widths=0.5, flierprops={"markersize": 2})
/tmp/ipykernel_181896/4084855955.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(X_train[col].dropna(), vert=True, widths=0.5, flierprops={"markersize": 2})
/tmp/ipykernel_181896/4084855955.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Us

,fitur,p1,median,p99,maks,pct_di_atas_p99
0,loan_amount,55000.000000,435000.000000,1.705000e+06,1.600500e+07,0.981796
1,income,30.000000,137.000000,8.269500e+02,7.780000e+04,0.988614
2,property_value,65000.000000,565000.000000,2.505000e+06,4.200500e+07,0.954524
3,ltv_calc,20.728519,80.246914,1.000000e+02,8.777778e+02,0.631804
4,dti_mid,15.000000,41.000000,6.500000e+01,6.500000e+01,0.000000
5,loan_term,180.000000,360.000000,3.600000e+02,4.800000e+02,0.297721
6,tract_to_msa_income_percentage,0.000000,107.000000,2.020000e+02,2.830000e+02,0.970432
7,ffiec_msa_md_median_family_income,76900.000000,130400.000000,1.484000e+05,1.484000e+05,0.000000
8,loan_to_income,0.585106,3.089838,7.862615e+00,1.225000e+03,0.988614
9,property_to_income,0.797872,3.963415,1.382159e+01,1.755000e+03,0.970432


In [14]:
corr = X_train[num].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
labels = [LBL[c] for c in num]
sns.heatmap(corr, annot=corr.map(lambda v: fmt_id(v, 2)), fmt="", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax,
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 7}, cbar_kws={"label": "Korelasi Spearman"})
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
plt.setp(ax.get_yticklabels(), fontsize=7)
save_figure(fig, "f3_corr_heatmap")
corr.round(2)

,loan_amount,income,property_value,ltv_calc,dti_mid,loan_term,tract_to_msa_income_percentage,ffiec_msa_md_median_family_income,loan_to_income,property_to_income
loan_amount,1.00,0.75,0.85,-0.00,-0.06,0.42,0.43,0.27,0.20,0.05
income,0.75,1.00,0.73,-0.13,-0.40,0.29,0.39,0.21,-0.41,-0.40
property_value,0.85,0.73,1.00,-0.42,-0.15,0.36,0.52,0.26,0.05,0.22
ltv_calc,-0.00,-0.13,-0.42,1.00,0.22,-0.18,-0.27,-0.04,0.14,-0.39
dti_mid,-0.06,-0.40,-0.15,0.22,1.00,-0.13,-0.10,-0.02,0.50,0.34
loan_term,0.42,0.29,0.36,-0.18,-0.13,1.00,0.21,0.12,0.30,0.30
tract_to_msa_income_percentage,0.43,0.39,0.52,-0.27,-0.10,0.21,1.00,-0.02,-0.01,0.11
ffiec_msa_md_median_family_income,0.27,0.21,0.26,-0.04,-0.02,0.12,-0.02,1.00,0.05,0.05
loan_to_income,0.20,-0.41,0.05,0.14,0.50,0.30,-0.01,0.05,1.00,0.78
property_to_income,0.05,-0.40,0.22,-0.39,0.34,0.30,0.11,0.05,0.78,1.00


## 8. Catatan keputusan preprocessing

In [15]:
D = [
    ("Target", "action_taken 1/2/3; kode lain tidak relevan", "y = 1 bila 3 (ditolak), y = 0 bila 1 atau 2", "Hanya kode 1–3 adalah keputusan kredit atas aplikasi penuh"),
    ("Kolom pasca-keputusan (denial_reason-1..4, interest_rate, rate_spread, hoepa_status, total_loan_costs, total_points_and_fees, origination_charges, discount_points, lender_credits, purchaser_type)", "Baru terisi setelah keputusan; banyak kosong untuk aplikasi ditolak", "Dibuang", "Leakage langsung: tidak tersedia saat aplikasi masuk"),
    ("loan_to_value_ratio (LTV laporan)", "Kosong 52,04% pada aplikasi ditolak vs 2,52% disetujui; tingkat tolak 73% bila kosong (t3_leakage_test, t3_ltv_replacement)", "Dibuang dan diganti ltv_calc = loan_amount / property_value × 100", "Pola kosong terjadi karena lender tidak menghitung LTV untuk aplikasi yang ditolak lebih awal, sehingga 'kosong' membocorkan keputusan. Informasi LTV sendiri penting bagi underwriting, dan dua komponennya (jumlah pinjaman yang diajukan dan nilai properti) sudah diketahui saat aplikasi masuk. ltv_calc berkorelasi tinggi dengan LTV laporan dan selisih pola kosongnya di bawah ambang 20 poin persen"),
    ("preapproval", "Kode 1 (pra-persetujuan diminta): 2.298 aplikasi, 0% ditolak (t3_leakage_code_test)", "Dibuang", "Artefak aturan pelaporan HMDA: pra-persetujuan yang ditolak dilaporkan sebagai action_taken 7 (di luar scope), sehingga kode 1 secara mekanis tidak pernah berlabel ditolak. Model akan mempelajari aturan pelaporan, bukan risiko pemohon"),
    ("aus-1", "Kode 6 'tidak berlaku/tidak memakai AUS': 9.056 aplikasi, tingkat tolak ±46% vs ±5% untuk kode AUS lain", "Dibuang", "Kode 6 umumnya muncul karena aplikasi dihentikan/ditolak sebelum dievaluasi AUS, sehingga mencerminkan jalannya proses (pasca-keputusan) dan bukan karakteristik pemohon. Hasil AUS sendiri tidak tersedia di data publik"),
    ("initially_payable_to_institution", "Kode 3 'tidak berlaku': 681 aplikasi, tingkat tolak ±81%", "Dibuang", "Kode 'tidak berlaku' terisi ketika tidak ada kewajiban kredit yang terbentuk (aplikasi tidak dicairkan), sehingga membocorkan hasil keputusan"),
    ("Atribut protected (derived_race/ethnicity/sex, applicant_age, kolom ras/etnis/jenis kelamin rinci)", "Atribut dilindungi fair lending", "Dikeluarkan dari fitur; derived_race, derived_ethnicity, derived_sex, applicant_age disimpan di sensitive_*.parquet untuk uji fairness", "Mencegah diskriminasi langsung"),
    ("tract_minority_population_percent", "Persentase penduduk minoritas di tract", "Dikeluarkan dari fitur", "Potensi proxy ras (diskriminasi tidak langsung)"),
    ("lei, census_tract, county_code, state_code, activity_year", "Identitas lender/lokasi dengan kardinalitas tinggi atau konstan", "Dibuang dari fitur", "Model dipakai satu bank (lei tidak relevan); lokasi rinci rawan proxy dan overfitting"),
    ("derived_msa-md", "Kode wilayah MSA", "Disimpan di meta_*.parquet hanya untuk analisis segmen", "Bukan prediktor; berguna untuk uji robustness per wilayah"),
    ("String 'NA', 'Exempt', '' pada semua kolom", "Kode kosong/pengecualian HMDA", "Diubah menjadi NaN", "Diperlakukan sebagai missing lalu diimputasi di pipeline"),
    ("Kode 1111 pada kolom kategorikal", "Exempt", "Diubah menjadi NaN", "Tidak memuat informasi kategori"),
    ("income ≤ 0", "Nilai tidak wajar (lihat t3_data_quality)", "Diubah menjadi NaN", "Pendapatan non-positif tidak masuk akal untuk KPR; mencegah rasio tak hingga"),
    ("debt_to_income_ratio", "Rentang string (<20%, 20%-<30%, 30%-<36%, 36–49, 50%-60%, >60%)", "Dipetakan ke nilai tengah: 15, 25, 33, 36–49, 55, 65 (dti_mid)", "Mempertahankan urutan ordinal; rentang terbuka diberi nilai tengah wajar"),
    ("co-applicant_age", "9999 = tidak ada co-applicant", "Fitur has_coapplicant (0/1); kolom usia sendiri tidak dipakai (protected)", "Keberadaan co-applicant relevan bagi kapasitas bayar tanpa memakai usia"),
    ("Fitur turunan", "-", "loan_to_income = loan_amount/(income×1.000); property_to_income = property_value/(income×1.000); ltv_calc", "Rasio beban kredit terhadap pendapatan dan agunan adalah logika inti underwriting"),
    ("Pola kosong property_value, income, DTI (uji lift)", "Jarang kosong (≤ 3,1%) tetapi tingkat tolak saat kosong jauh di atas rata-rata (lift ≥ 2; lihat kolom lift di t3_leakage_test)", "Nilai tetap dipakai, tetapi status 'kosong' TIDAK dijadikan fitur (tanpa missing indicator); aplikasi berdata tidak lengkap ditangani aturan bisnis data_tidak_lengkap (Tahap 8)", "Kosong terjadi karena lender berhenti memproses aplikasi yang ditolak lebih awal (mekanisme sama dengan LTV laporan) — informasi pasca-keputusan; pada aplikasi baru nilai ini selalu diisi pemohon"),
    ("Missing value numerik", "Kosong ≤ 3,1% per fitur (t3_missing_summary)", "Imputasi median di dalam Pipeline (fit di train), tanpa missing indicator", "Median robust terhadap skew; indikator kosong dihapus karena membocorkan keputusan (baris di atas)"),
    ("Missing value kategorikal", "Sangat sedikit", "Imputasi modus + OneHotEncoder(handle_unknown='ignore') di Pipeline", "Aman terhadap kategori baru pada test/OOT"),
    ("Outlier & skew (loan_amount, income, property_value)", "Distribusi sangat menceng kanan (f3_dist_before_after_log, t3_outlier_percentiles)", "Capping persentil 1–99 (QuantileCapper, fit di train) lalu log1p", "Mengurangi pengaruh nilai ekstrem tanpa membuang baris"),
    ("Outlier fitur numerik lain", "Ekor panjang pada rasio (f3_boxplot_numerik)", "Capping persentil 1–99 tanpa log", "Konsisten antar-fitur dan tetap di dalam pipeline"),
    ("Skala", "Rentang fitur sangat berbeda", "StandardScaler di pipeline", "Wajib untuk Logistic Regression; tidak berpengaruh pada tree"),
    ("Duplikat persis", "Lihat t3_data_quality", "Dipertahankan", "Tanpa ID aplikasi, baris identik bisa merupakan aplikasi berbeda (nilai dibulatkan/dibinning)"),
    ("Split", "Data 2024", "Stratified 80/20 train/test, random_state 42; 2025 utuh sebagai OOT", "Test untuk evaluasi akhir, OOT untuk generalisasi lintas waktu"),
    ("Imbalance", "±11,75% ditolak (t3_class_balance)", "Diputuskan di Tahap 5 (class weighting utama, SMOTE pembanding)", "Penanganan harus di dalam CV agar tidak bocor"),
    ("Teknik tidak dipakai: penghapusan baris missing", "-", "Tidak dipakai", "Akan membuang aplikasi ditolak secara tidak proporsional (bias)"),
    ("Teknik tidak dipakai: winsorizing/penghapusan outlier di luar pipeline", "-", "Tidak dipakai", "Batas harus dipelajari dari train saja; dilakukan oleh QuantileCapper"),
    ("Teknik tidak dipakai: PCA/reduksi dimensi", "Hanya 13 fitur", "Tidak dipakai", "Jumlah fitur kecil dan interpretabilitas lebih penting"),
    ("Teknik tidak dipakai: target encoding", "Kardinalitas kategori rendah", "Tidak dipakai", "One-hot cukup dan tanpa risiko leakage target"),
]
decisions = pd.DataFrame(D, columns=["kolom_aspek", "temuan", "keputusan", "alasan"])
decisions.insert(0, "no", range(1, len(decisions) + 1))
save_table(decisions, "t3_decisions")
decisions

,no,kolom_aspek,temuan,keputusan,alasan
0,1,Target,action_taken 1/2/3; kode lain tidak relevan,"y = 1 bila 3 (ditolak), y = 0 bila 1 atau 2",Hanya kode 1–3 adalah keputusan kredit atas ap...
1,2,"Kolom pasca-keputusan (denial_reason-1..4, int...",Baru terisi setelah keputusan; banyak kosong u...,Dibuang,Leakage langsung: tidak tersedia saat aplikasi...
2,3,loan_to_value_ratio (LTV laporan),"Kosong 52,04% pada aplikasi ditolak vs 2,52% d...",Dibuang dan diganti ltv_calc = loan_amount / p...,Pola kosong terjadi karena lender tidak menghi...
3,4,preapproval,Kode 1 (pra-persetujuan diminta): 2.298 aplika...,Dibuang,Artefak aturan pelaporan HMDA: pra-persetujuan...
4,5,aus-1,Kode 6 'tidak berlaku/tidak memakai AUS': 9.05...,Dibuang,Kode 6 umumnya muncul karena aplikasi dihentik...
5,6,initially_payable_to_institution,"Kode 3 'tidak berlaku': 681 aplikasi, tingkat ...",Dibuang,Kode 'tidak berlaku' terisi ketika tidak ada k...
6,7,"Atribut protected (derived_race/ethnicity/sex,...",Atribut dilindungi fair lending,"Dikeluarkan dari fitur; derived_race, derived_...",Mencegah diskriminasi langsung
7,8,tract_minority_population_percent,Persentase penduduk minoritas di tract,Dikeluarkan dari fitur,Potensi proxy ras (diskriminasi tidak langsung)
8,9,"lei, census_tract, county_code, state_code, ac...",Identitas lender/lokasi dengan kardinalitas ti...,Dibuang dari fitur,Model dipakai satu bank (lei tidak relevan); l...
9,10,derived_msa-md,Kode wilayah MSA,Disimpan di meta_*.parquet hanya untuk analisi...,Bukan prediktor; berguna untuk uji robustness ...
